# STED Depletion Experiment: Projection-Based Subspace Removal
*Tests if removing the primary probe direction collapses the typosquat subspace.*

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import json, random, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from google.colab import files
import zipfile, io
uploaded = files.upload()
fname = list(uploaded.keys())[0]

if fname.endswith('.zip'):
    with zipfile.ZipFile(io.BytesIO(uploaded[fname])) as zf:
        jsonl = [n for n in zf.namelist() if n.endswith('.jsonl')][0]
        zf.extract(jsonl)
        DATASET_PATH = jsonl
else:
    DATASET_PATH = fname
print(f"Using: {DATASET_PATH}")

df = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df = df[df['is_adversarial'] == True].copy()
print(f"Loaded {len(df)} adversarial entries")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto" if device.type=="cuda" else None)
if device.type != "cuda": model.to(device)
model.eval()

In [ ]:
def find_package_span(cmd, pkg):
    m = re.search(r'\b' + re.escape(pkg) + r'\b', cmd)
    return (m.start(), m.end()) if m else None

def char_to_token_span(tokenizer, text, cs, ce):
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=True)
    offs = enc['offset_mapping']
    ts = te = None
    for i, (s, e) in enumerate(offs):
        if s <= cs < e or s < ce <= e:
            if ts is None: ts = i
            te = i
    return (ts or len(offs)-1, te or len(offs)-1)

def extract_hidden_states(model, tokenizer, commands, packages, layers=(12,16), batch_size=8):
    model.eval()
    all_states = []
    for i in range(0, len(commands), batch_size):
        batch_cmds = commands[i:i+batch_size]
        batch_pkgs = packages[i:i+batch_size]
        inputs = tokenizer(batch_cmds, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states
        for j, (cmd, pkg) in enumerate(zip(batch_cmds, batch_pkgs)):
            span = find_package_span(cmd, pkg)
            if span is None:
                states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            else:
                ts, te = char_to_token_span(tokenizer, cmd, *span)
                if ts > te or ts >= hidden[0].shape[1]:
                    states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
                else:
                    states = [hidden[l][j, ts:te+1, :].float().cpu().numpy().mean(axis=0) for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            if states:
                combined = np.concatenate(states)
                combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0)
                all_states.append(combined)
    return np.array(all_states) if all_states else None

texts, pkgs, labels = [], [], []
for _, row in df.iterrows():
    texts.append(row['clean_command']); pkgs.append(row['package_name']); labels.append(0)
    texts.append(row['typo_command']); pkgs.append(row['typo_package']); labels.append(1)

print("Extracting features...")
X_base = extract_hidden_states(model, tokenizer, texts, pkgs)
y = np.array(labels)

In [ ]:
probe = LogisticRegression(max_iter=1000, random_state=SEED)
probe.fit(X_base, y)
w_probe = probe.coef_[0]
auc_base = roc_auc_score(y, probe.predict_proba(X_base)[:, 1])
print(f"Baseline AUC: {auc_base:.4f}")

In [ ]:
def depletion_projection(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    w_unit = w / np.linalg.norm(w)
    projection = np.outer(X @ w_unit, w_unit)
    return X - projection

X_depleted = depletion_projection(X_base, w_probe)

probe_depleted = LogisticRegression(max_iter=1000, random_state=SEED)
probe_depleted.fit(X_depleted, y)
auc_depleted = roc_auc_score(y, probe_depleted.predict_proba(X_depleted)[:, 1])

try:
    w_depleted = probe_depleted.coef_[0]
    cos_sim = np.dot(w_probe, w_depleted) / (np.linalg.norm(w_probe) * np.linalg.norm(w_depleted))
except:
    cos_sim = float('nan')

print(f"Depleted AUC: {auc_depleted:.4f}")
print(f"Direction Cosine: {cos_sim:.4f}")
print(f"\nBaseline AUC: {auc_base:.4f}")
print(f"Depleted AUC: {auc_depleted:.4f}")
print(f"AUC Drop: {auc_base - auc_depleted:.4f}")

if auc_depleted < 0.6:
    print("✓ SUCCESS: Subspace collapsed! The separation lived primarily along w.")
else:
    print("⚠ Subspace persists. Multi-direction depletion or contrastive FT needed.")